# Progress-Gated Nearest-Analog Correction

Fourth alignment-track experiment. The pseudo-zone residual-head screen showed analog transfer is most useful in later target-zone rows. This notebook tests simple progress-gated shrinkage schedules for the nearest-analog delta gap.

## 1. Setup

In [1]:
"Configure imports, paths, and progress-gated analog schedules."

from pathlib import Path
import itertools
import json
import shutil
import time

import numpy as np
import pandas as pd
from sklearn.metrics import mean_absolute_error, mean_squared_error

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "context").exists():
    PROJECT_ROOT = next(parent for parent in PROJECT_ROOT.parents if (parent / "context").exists())

DATE_STAMP = "2026-06-23"
RUN_ID = f"{DATE_STAMP}__progress_gated_analog"
ARTIFACT_TABLES = PROJECT_ROOT / "artifacts" / "tables"
ARTIFACT_REPORTS = PROJECT_ROOT / "artifacts" / "reports"
ARTIFACT_LOGS = PROJECT_ROOT / "artifacts" / "logs" / "runs" / RUN_ID
for path in [ARTIFACT_TABLES, ARTIFACT_REPORTS, ARTIFACT_LOGS]:
    path.mkdir(parents=True, exist_ok=True)

BASE_RESIDUAL_PATH = ARTIFACT_TABLES / "model__hsgp_residual_on_gam_huber_residual_sample__2026-06-22.csv"
ANALOG_SAMPLE_PATH = ARTIFACT_TABLES / "model__multi_analog_zone_blend_residual_sample__2026-06-23.csv"
TARGET_CANDIDATE = "huber_plus_hsgp_dps_m8"
ANALOG_SOURCE_CANDIDATE = "weighted_k1_s050"
LOCAL_REFERENCE_RMSE = 14.3586
PUBLIC_REFERENCE_RMSE = 14.514
OFFSET_CLIP = 12.0

SCHEDULES = []
SCHEDULES.append({"name": "baseline_no_gate", "z0": 0.0, "z1": 0.0, "z2": 0.0, "z3": 0.0, "family": "baseline"})
SCHEDULES.append({"name": "fixed_s005", "z0": 0.05, "z1": 0.05, "z2": 0.05, "z3": 0.05, "family": "fixed"})
SCHEDULES.append({"name": "fixed_s010", "z0": 0.10, "z1": 0.10, "z2": 0.10, "z3": 0.10, "family": "fixed"})
SCHEDULES.append({"name": "late_only_000_000_010_010", "z0": 0.00, "z1": 0.00, "z2": 0.10, "z3": 0.10, "family": "manual_late"})
SCHEDULES.append({"name": "ramp_000_025_075_100", "z0": 0.00, "z1": 0.025, "z2": 0.075, "z3": 0.10, "family": "manual_ramp"})
SCHEDULES.append({"name": "ramp_025_050_100_125", "z0": 0.025, "z1": 0.05, "z2": 0.10, "z3": 0.125, "family": "manual_ramp"})
for z0, z1, z2, z3 in itertools.product([0.0, 0.025, 0.05], [0.0, 0.025, 0.05, 0.075], [0.05, 0.075, 0.10, 0.125, 0.15], [0.05, 0.075, 0.10, 0.125, 0.15]):
    if z0 <= z1 <= z2 <= z3:
        SCHEDULES.append({"name": f"grid_{int(z0*1000):03d}_{int(z1*1000):03d}_{int(z2*1000):03d}_{int(z3*1000):03d}", "z0": z0, "z1": z1, "z2": z2, "z3": z3, "family": "monotone_grid"})

print(f"project_root={PROJECT_ROOT}")
print(f"run_id={RUN_ID}")
print(f"schedule_count={len(SCHEDULES)}")

project_root=C:\Users\sardo\OneDrive\Desktop\Classes\Projects\Wellbore-tvt-Modeling
run_id=2026-06-23__progress_gated_analog
schedule_count=126


## 2. Load Residuals And Nearest-Analog Gap

In [2]:
"Load the current best residual sample and nearest-analog delta gap."

base_all = pd.read_csv(BASE_RESIDUAL_PATH)
base = base_all.loc[base_all["candidate"].eq(TARGET_CANDIDATE)].copy()
for col in ["fold", "row_idx", "eval_progress", "y_true", "y_pred", "residual"]:
    base[col] = pd.to_numeric(base[col], errors="coerce")
base["fold"] = base["fold"].astype(int)
base["row_idx"] = base["row_idx"].astype(int)
base["baseline_pred"] = base["y_pred"]
base["baseline_residual"] = base["y_true"] - base["baseline_pred"]
base["pseudo_zone"] = pd.cut(base["eval_progress"], bins=[-0.001, 0.25, 0.50, 0.75, 1.001], labels=["z0", "z1", "z2", "z3"])

analog = pd.read_csv(ANALOG_SAMPLE_PATH)
analog = analog.loc[analog["candidate"].eq(ANALOG_SOURCE_CANDIDATE), ["fold", "wellname", "row_idx", "analog_gap"]].copy()
for col in ["fold", "row_idx", "analog_gap"]:
    analog[col] = pd.to_numeric(analog[col], errors="coerce")
analog["fold"] = analog["fold"].astype(int)
analog["row_idx"] = analog["row_idx"].astype(int)

model_frame = base.merge(analog, on=["fold", "wellname", "row_idx"], how="left")
model_frame["analog_gap_clipped"] = model_frame["analog_gap"].clip(-120.0, 120.0)
print(f"rows={len(model_frame):,}")
print(f"baseline_fold_mean_rmse={np.mean([np.sqrt(np.mean(g.baseline_residual.to_numpy() ** 2)) for _, g in model_frame.groupby('fold')]):.4f}")
print(model_frame.groupby("pseudo_zone", observed=False).size().rename("rows_by_zone").to_string())

rows=5,400
baseline_fold_mean_rmse=14.7385
pseudo_zone
z0    1160
z1    1414
z2    1412
z3    1414


## 3. Evaluate Gated Schedules

In [3]:
"Apply progress-gated analog shrinkage schedules and score by fold and zone."

start_time = time.time()
metric_rows = []
prediction_rows = []
zone_rows = []
for schedule in SCHEDULES:
    for fold, valid_rows in model_frame.groupby("fold"):
        zone_shrink = valid_rows["pseudo_zone"].astype(str).map({"z0": schedule["z0"], "z1": schedule["z1"], "z2": schedule["z2"], "z3": schedule["z3"]}).fillna(0.0).to_numpy(dtype=float)
        correction = np.clip(valid_rows["analog_gap_clipped"].fillna(0.0).to_numpy(dtype=float) * zone_shrink, -OFFSET_CLIP, OFFSET_CLIP)
        y_true = valid_rows["y_true"].to_numpy()
        y_pred = valid_rows["baseline_pred"].to_numpy() + correction
        residual_after = y_true - y_pred
        metric_rows.append(
            {
                "candidate": schedule["name"],
                "family": schedule["family"],
                "fold": int(fold),
                "rmse": float(np.sqrt(mean_squared_error(y_true, y_pred))),
                "mae": float(mean_absolute_error(y_true, y_pred)),
                "n_rows": int(len(valid_rows)),
                "n_wells": int(valid_rows["wellname"].nunique()),
                "mean_correction": float(np.mean(correction)),
                "std_correction": float(np.std(correction)),
                "max_abs_correction": float(np.max(np.abs(correction))) if len(correction) else 0.0,
                "z0": schedule["z0"],
                "z1": schedule["z1"],
                "z2": schedule["z2"],
                "z3": schedule["z3"],
            }
        )
        out = valid_rows[["fold", "wellname", "row_idx", "pseudo_zone", "y_true", "baseline_pred", "baseline_residual", "eval_progress", "analog_gap_clipped"]].copy()
        out["candidate"] = schedule["name"]
        out["zone_shrink"] = zone_shrink
        out["analog_correction"] = correction
        out["y_pred"] = y_pred
        out["residual_after_gate"] = residual_after
        prediction_rows.append(out)
        for zone, zone_frame in out.groupby("pseudo_zone", observed=False):
            zone_rows.append(
                {
                    "candidate": schedule["name"],
                    "fold": int(fold),
                    "pseudo_zone": str(zone),
                    "rmse_before": float(np.sqrt(np.mean(zone_frame["baseline_residual"].to_numpy() ** 2))),
                    "rmse_after": float(np.sqrt(np.mean(zone_frame["residual_after_gate"].to_numpy() ** 2))),
                    "rows": int(len(zone_frame)),
                    "mean_shrink": float(zone_frame["zone_shrink"].mean()),
                }
            )

fold_metrics = pd.DataFrame(metric_rows)
predictions = pd.concat(prediction_rows, ignore_index=True)
zone_metrics = pd.DataFrame(zone_rows)
summary = (
    fold_metrics.groupby(["candidate", "family"], as_index=False)
    .agg(
        rmse_mean=("rmse", "mean"),
        rmse_std=("rmse", "std"),
        mae_mean=("mae", "mean"),
        mean_correction=("mean_correction", "mean"),
        std_correction=("std_correction", "mean"),
        max_abs_correction=("max_abs_correction", "max"),
        z0=("z0", "first"),
        z1=("z1", "first"),
        z2=("z2", "first"),
        z3=("z3", "first"),
        n_rows=("n_rows", "sum"),
        n_wells=("n_wells", "sum"),
    )
)
baseline_rmse = float(summary.loc[summary["candidate"].eq("baseline_no_gate"), "rmse_mean"].iloc[0])
summary["rmse_delta_vs_sample_baseline"] = summary["rmse_mean"] - baseline_rmse
summary = summary.sort_values(["rmse_mean", "candidate"]).reset_index(drop=True)
train_time_s = time.time() - start_time
print(summary.head(20).to_string(index=False))

                candidate        family  rmse_mean  rmse_std  mae_mean  mean_correction  std_correction  max_abs_correction    z0    z1    z2    z3  n_rows  n_wells  rmse_delta_vs_sample_baseline
     grid_025_025_075_100 monotone_grid  14.696574  0.378961 10.379576         0.023016        1.222216            8.029752 0.025 0.025 0.075 0.100    5400      773                      -0.041971
     grid_000_025_075_100 monotone_grid  14.696725  0.381584 10.381224         0.022483        1.215178            8.029752 0.000 0.025 0.075 0.100    5400      773                      -0.041820
     ramp_000_025_075_100   manual_ramp  14.696725  0.381584 10.381224         0.022483        1.215178            8.029752 0.000 0.025 0.075 0.100    5400      773                      -0.041820
     grid_000_000_075_100 monotone_grid  14.696933  0.391215 10.379760         0.020913        1.199065            8.029752 0.000 0.000 0.075 0.100    5400      773                      -0.041612
     grid_025_025_07

## 4. Diagnostics And Export

In [4]:
"Summarize the best gated schedule and export artifacts."

best_candidate = summary.iloc[0]["candidate"]
best_row = summary.iloc[0]
baseline_row = summary.loc[summary["candidate"].eq("baseline_no_gate")].iloc[0]
best_predictions = predictions.loc[predictions["candidate"].eq(best_candidate)].copy()
best_zone = zone_metrics.loc[zone_metrics["candidate"].eq(best_candidate)].copy()
zone_summary = (
    best_zone.groupby("pseudo_zone", as_index=False)
    .agg(
        rmse_before=("rmse_before", "mean"),
        rmse_after=("rmse_after", "mean"),
        rows=("rows", "sum"),
        mean_shrink=("mean_shrink", "mean"),
    )
)
zone_summary["rmse_delta"] = zone_summary["rmse_after"] - zone_summary["rmse_before"]

well_before = model_frame.groupby(["fold", "wellname"], as_index=False).agg(mean_residual_before=("baseline_residual", "mean"))
well_after = best_predictions.groupby(["fold", "wellname"], as_index=False).agg(mean_residual_after=("residual_after_gate", "mean"), mean_analog_correction=("analog_correction", "mean"))
well_diagnostics = well_after.merge(well_before, on=["fold", "wellname"], how="left")
well_diagnostics["abs_bias_delta"] = well_diagnostics["mean_residual_after"].abs() - well_diagnostics["mean_residual_before"].abs()

findings = pd.DataFrame(
    [
        {"finding": "best_candidate", "value": f"{best_candidate} RMSE {best_row['rmse_mean']:.4f} vs sample baseline {baseline_row['rmse_mean']:.4f}", "modeling_implication": "Promote only if progress-gated analog correction clears the 0.05 RMSE gate."},
        {"finding": "best_schedule", "value": f"z0={best_row['z0']:.3f}, z1={best_row['z1']:.3f}, z2={best_row['z2']:.3f}, z3={best_row['z3']:.3f}", "modeling_implication": "Shows how strongly to trust nearest analogs by evaluation progress."},
        {"finding": "sample_baseline_delta", "value": f"best candidate delta {best_row['rmse_delta_vs_sample_baseline']:.4f}", "modeling_implication": "If this beats fixed 0.10 shrink, use it as the next Kaggle-safe analog schedule candidate."},
        {"finding": "well_bias_change", "value": f"median absolute well-bias delta {well_diagnostics['abs_bias_delta'].median():.4f}", "modeling_implication": "Negative is good; positive means the schedule adds well-level noise."},
    ]
)

summary_path = ARTIFACT_TABLES / f"model__progress_gated_analog_summary__{DATE_STAMP}.csv"
fold_path = ARTIFACT_TABLES / f"model__progress_gated_analog_fold_metrics__{DATE_STAMP}.csv"
pred_path = ARTIFACT_TABLES / f"model__progress_gated_analog_residual_sample__{DATE_STAMP}.csv"
well_path = ARTIFACT_TABLES / f"model__progress_gated_analog_well_diagnostics__{DATE_STAMP}.csv"
zone_path = ARTIFACT_TABLES / f"model__progress_gated_analog_zone_summary__{DATE_STAMP}.csv"
findings_path = ARTIFACT_TABLES / f"model__progress_gated_analog_findings__{DATE_STAMP}.csv"
report_path = ARTIFACT_REPORTS / f"model__progress_gated_analog__{DATE_STAMP}.md"
comparison_path = ARTIFACT_TABLES / "model_comparison.csv"

summary.to_csv(summary_path, index=False)
fold_metrics.to_csv(fold_path, index=False)
predictions.to_csv(pred_path, index=False)
well_diagnostics.to_csv(well_path, index=False)
zone_summary.to_csv(zone_path, index=False)
findings.to_csv(findings_path, index=False)

report = f"""# Progress-Gated Nearest-Analog Correction

Run ID: `{RUN_ID}`

Result

Best candidate: `{best_candidate}`

- Same-sample baseline RMSE: **{baseline_row['rmse_mean']:.4f}**
- Best RMSE: **{best_row['rmse_mean']:.4f}**
- Delta vs same-sample baseline: **{best_row['rmse_delta_vs_sample_baseline']:.4f}**
- Schedule: z0 **{best_row['z0']:.3f}**, z1 **{best_row['z1']:.3f}**, z2 **{best_row['z2']:.3f}**, z3 **{best_row['z3']:.3f}**
- Current full local reference: **{LOCAL_REFERENCE_RMSE:.4f}**
- Current public reference: **{PUBLIC_REFERENCE_RMSE:.3f}**

Interpretation

This screen tests whether analog correction should be gated by evaluation progress. It is still a residual-sample diagnostic, not a Kaggle submission notebook.
"""
report_path.write_text(report, encoding="utf-8")

comparison_row = pd.DataFrame(
    [
        {
            "run_id": RUN_ID,
            "model_family": "alignment_progress_gate",
            "model_name": best_candidate,
            "cv_rmse_mean": float(best_row["rmse_mean"]),
            "cv_rmse_std": float(best_row["rmse_std"]),
            "train_time_s": float(train_time_s),
            "n_features": 4,
            "notes": "Fold-safe progress-gated nearest-analog shrinkage schedule over Huber + additive Hilbert residual sample",
        }
    ]
)
comparison = pd.read_csv(comparison_path) if comparison_path.exists() else pd.DataFrame()
if len(comparison):
    comparison = comparison.loc[~comparison["run_id"].eq(RUN_ID)]
comparison = pd.concat([comparison, comparison_row], ignore_index=True)
comparison.to_csv(comparison_path, index=False)

for src in [summary_path, fold_path, pred_path, well_path, zone_path, findings_path, report_path, comparison_path]:
    shutil.copy2(src, ARTIFACT_LOGS / src.name)

config = {"run_id": RUN_ID, "target_candidate": TARGET_CANDIDATE, "analog_source_candidate": ANALOG_SOURCE_CANDIDATE, "offset_clip": OFFSET_CLIP, "schedules": SCHEDULES, "local_reference_rmse": LOCAL_REFERENCE_RMSE, "public_reference_rmse": PUBLIC_REFERENCE_RMSE}
(ARTIFACT_LOGS / "config.json").write_text(json.dumps(config, indent=2), encoding="utf-8")

print(findings.to_string(index=False))
print(zone_summary.to_string(index=False))
print(f"wrote {summary_path}")
print(f"wrote run bundle {ARTIFACT_LOGS}")

              finding                                                        value                                                                       modeling_implication
       best_candidate grid_025_025_075_100 RMSE 14.6966 vs sample baseline 14.7385                Promote only if progress-gated analog correction clears the 0.05 RMSE gate.
        best_schedule                       z0=0.025, z1=0.025, z2=0.075, z3=0.100                        Shows how strongly to trust nearest analogs by evaluation progress.
sample_baseline_delta                                 best candidate delta -0.0420 If this beats fixed 0.10 shrink, use it as the next Kaggle-safe analog schedule candidate.
     well_bias_change                      median absolute well-bias delta -0.0888                       Negative is good; positive means the schedule adds well-level noise.
pseudo_zone  rmse_before  rmse_after  rows  mean_shrink  rmse_delta
         z0     7.872198    7.870954  1160        0.025   -0.0

## 5. Conclusion

In [5]:
"Print final decision summary."

print(f"best_candidate={best_candidate}")
print(f"sample_baseline_rmse={baseline_row['rmse_mean']:.4f}")
print(f"best_rmse={best_row['rmse_mean']:.4f}")
print(f"delta_vs_sample_baseline={best_row['rmse_delta_vs_sample_baseline']:.4f}")
print(f"promotion_gate_cleared={best_row['rmse_delta_vs_sample_baseline'] <= -0.05}")

best_candidate=grid_025_025_075_100
sample_baseline_rmse=14.7385
best_rmse=14.6966
delta_vs_sample_baseline=-0.0420
promotion_gate_cleared=False
